# Derivatives and Numerical Gradients

**Checkpoint:** CP0.2, Notebook 01  
**Goal:** connect derivatives and partial derivatives to gradient vectors, then verify analytical gradients with finite differences.

Run the notebook from top to bottom. Before each code cell, predict what it should print.

## 1. Derivatives of one-variable functions

The derivative of `f` at `x` is the instantaneous rate of change of the function. It is the limit of the slope of a secant line:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h)-f(x)}{h}.$$

For `f(x) = x^2 + 3x + 2`, the power rule gives `f'(x) = 2x + 3`. At `x = 2`, the derivative is `7`.

In [ ]:
def f(x):
    return x**2 + 3 * x + 2


def analytical_gradient(x):
    return 2 * x + 3


def numerical_gradient(x, eps=1e-5):
    """Estimate the derivative with a centered finite difference."""
    return (f(x + eps) - f(x - eps)) / (2 * eps)


x = 2.0
analytical = analytical_gradient(x)
numerical = numerical_gradient(x)

print(f"Analytical derivative at x={x}: {analytical:.10f}")
print(f"Numerical derivative at x={x}:  {numerical:.10f}")
assert abs(analytical - numerical) < 1e-8

The centered formula evaluates the function on both sides of `x`. Its leading truncation error is usually smaller than the one-sided difference `(f(x + eps) - f(x)) / eps`. It is still an approximation: `eps` must be small enough to approximate the limit, but not so small that floating-point rounding dominates.

## 2. Multivariable derivatives and partial derivatives

For a function of several variables, a partial derivative measures how the output changes as one input changes while the others are held fixed. The gradient collects all partial derivatives into a vector.

Use `h(x, y) = x^2 + 3xy + 2y^2`. Its gradient is `grad h(x, y) = [2x + 3y, 3x + 4y]`.

In [ ]:
import numpy as np


def h(point):
    x, y = point
    return x**2 + 3 * x * y + 2 * y**2


def analytical_gradient_2d(point):
    x, y = point
    return np.array([2 * x + 3 * y, 3 * x + 4 * y], dtype=float)


def numerical_gradient_vector(function, point, eps=1e-5):
    """Estimate each gradient component with centered differences."""
    point = np.asarray(point, dtype=float)
    gradient = np.zeros_like(point)

    for index in range(point.size):
        offset = np.zeros_like(point)
        offset[index] = eps
        gradient[index] = (
            function(point + offset) - function(point - offset)
        ) / (2 * eps)

    return gradient


test_points = [np.array([1.0, 2.0]), np.array([-0.5, 0.25]), np.array([3.0, -1.0])]
for point in test_points:
    analytical = analytical_gradient_2d(point)
    numerical = numerical_gradient_vector(h, point)
    print(f"point={point}, analytical={analytical}, numerical={numerical}")
    np.testing.assert_allclose(analytical, numerical, rtol=1e-7, atol=1e-7)

The assertion checks every gradient component at several points. A successful assertion means the values agree within the stated relative and absolute tolerances; it does not prove that the analytical formula is correct for every possible input.

## 3. How does `eps` affect the estimate?

Compare a range of step sizes for the one-variable example. The exact derivative at `x = 2` is known, so we can measure absolute error directly.

In [ ]:
exact = analytical_gradient(x)
for eps in [1e-1, 1e-3, 1e-5, 1e-7, 1e-9]:
    estimate = numerical_gradient(x, eps=eps)
    error = abs(estimate - exact)
    print(f"eps={eps:.0e}: estimate={estimate:.12f}, absolute error={error:.3e}")

For this quadratic, the centered-difference formula is exact in real arithmetic. Any remaining discrepancy comes from floating-point arithmetic. For more general smooth functions, truncation error also matters: decreasing `eps` first tends to improve the estimate, but eventually rounding and cancellation can make it worse.

## Exercises

1. Derive the gradient of `q(x, y) = 3x^2 - 2xy + y^2` by hand.
2. Implement the analytical gradient and compare it with `numerical_gradient_vector` at three points.
3. Why can a finite-difference gradient be useful when the analytical derivative is difficult to derive? What can it not guarantee?

### Answer key (read after attempting)

The gradient is `[6x - 2y, -2x + 2y]`. The numerical comparison should use a stated tolerance and multiple points. Finite differences are useful as a numerical check, but they are approximate and can be affected by step size and floating-point error.

In [ ]:
def q(point):
    x, y = point
    return 3 * x**2 - 2 * x * y + y**2


def analytical_gradient_q(point):
    x, y = point
    return np.array([6 * x - 2 * y, -2 * x + 2 * y], dtype=float)


for point in [np.array([1.0, 2.0]), np.array([-2.0, 0.5]), np.array([0.0, 0.0])]:
    np.testing.assert_allclose(
        analytical_gradient_q(point),
        numerical_gradient_vector(q, point),
        rtol=1e-7,
        atol=1e-7,
    )
print("All exercise gradient checks passed.")